# 03 — Train & SageMaker

1. **Model comparison** (Linear Regression, Random Forest default/tuned, XGBoost) — same features as `src/train.py`.
2. **Cloud training** — configure `config/` and AWS, upload data, submit job.

Dev dependencies: `pip install -r ../requirements-dev.txt`

In [ ]:
import sys
from pathlib import Path

ROOT = Path("..").resolve()
SRC = ROOT / "src"
sys.path.insert(0, str(SRC))

from compare_models import run_compare

run_compare(str(ROOT / "data" / "raw" / "hour.csv"), test_size_ratio=0.2, output_json=str(ROOT / "models" / "compare_metrics.json"))

## Diagnostics (XGBoost holdout)

Same time-based split as the comparison table: actual vs predicted and residuals.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from xgboost import XGBRegressor

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))
from preprocess import build_training_matrices, load_hour_csv

df = load_hour_csv(str(ROOT / "data" / "raw" / "hour.csv"))
X_train, X_test, y_train, y_test, *_ = build_training_matrices(df, test_size_ratio=0.2)
xgb = XGBRegressor(
    n_estimators=700,
    learning_rate=0.03,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
)
xgb.fit(X_train, y_train)
y_pred = xgb.predict(X_test)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(y_test, y_pred, alpha=0.3)
axes[0].set_xlabel("Actual cnt")
axes[0].set_ylabel("Predicted cnt")
axes[0].set_title("XGBoost: actual vs predicted")

residuals = y_test - y_pred
axes[1].scatter(y_pred, residuals, alpha=0.3)
axes[1].axhline(0, color="red")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("Residual")
axes[1].set_title("Residual plot")
plt.tight_layout()
plt.show()

## SageMaker training job

Set `SAGEMAKER_ROLE_ARN`, `SAGEMAKER_BUCKET`, upload `hour.csv` to S3 (`scripts/upload_data.sh`), then uncomment:

In [ ]:
import sys
from pathlib import Path

cfg = Path("../config").resolve()
sys.path.insert(0, str(cfg))

from sagemaker_config import submit_training_job

# submit_training_job()